## Import statements

In [1625]:
import tensorflow as tf 
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import plotly
import plotly.express as px


In [1626]:
# load dataset, set date column as index
# df = pd.read_csv('../../datasets/TX-Data/soil_station/SM_4.dat', sep=",", parse_dates=["Date"],  index_col="Date") 
df = pd.read_csv('../../datasets/TX-Data/MET_station/MET_4.dat', sep=",", parse_dates=["Date"],  index_col="Date") 

# manually converting date column to datetime
df.index = pd.to_datetime(df.index, format='%m/%d/%y %H:%M')

print(df.info())

/var/folders/kf/3zgjks2j09s7fg110wp13m6h0000gn/T/ipykernel_20376/3466925635.py:3: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 60649 entries, 2014-10-01 00:00:00 to 2021-09-01 00:00:00
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0    Ppt             60649 non-null  object
 1    Tair            60649 non-null  object
 2    RH              60649 non-null  object
 3    Wind speed      60649 non-null  object
 4    Wind direction  60649 non-null  object
 5    Srad            60649 non-null  object
dtypes: object(6)
memory usage: 3.2+ MB
None


In [1627]:
print(df.isnull().sum()) # none?? wrong, look into
df.dropna() # still doing it anyways

Ppt               0
Tair              0
RH                0
Wind speed        0
Wind direction    0
Srad              0
dtype: int64


,Ppt,Tair,RH,Wind speed,Wind direction,Srad
Date,,,,,,
2014-10-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2021-08-31 20:00:00,0.000,29.940,52.58,1.237,149.70,0.06
2021-08-31 21:00:00,0.000,28.880,55.09,1.612,160.10,0.00
2021-08-31 22:00:00,0.000,28.190,57.87,2.043,157.20,0.00


In [1628]:
df.shape

(60649, 6)

In [1629]:
df.head(25) 
# collecting data every hour from hr 0 to 23

,Ppt,Tair,RH,Wind speed,Wind direction,Srad
Date,,,,,,
2014-10-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 05:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN


Gathered information...

SM Columns: 
- Date (MM/DD/YY HR:MIN)
- ppt (precipitation)
- SWC_5, SWC_10, SWC_20, SWC_50 (soil water content going deep in centimeters)
- T_5, T_10, T_20, T_50 (time)
Flag

Collects data every hour in the day
Data shape: (59161, 10)


### just a bit of data Cleaning now

In [1630]:
df.index

DatetimeIndex(['2014-10-01 00:00:00', '2014-10-01 01:00:00',
               '2014-10-01 02:00:00', '2014-10-01 03:00:00',
               '2014-10-01 04:00:00', '2014-10-01 05:00:00',
               '2014-10-01 06:00:00', '2014-10-01 07:00:00',
               '2014-10-01 08:00:00', '2014-10-01 09:00:00',
               ...
               '2021-08-31 15:00:00', '2021-08-31 16:00:00',
               '2021-08-31 17:00:00', '2021-08-31 18:00:00',
               '2021-08-31 19:00:00', '2021-08-31 20:00:00',
               '2021-08-31 21:00:00', '2021-08-31 22:00:00',
               '2021-08-31 23:00:00', '2021-09-01 00:00:00'],
              dtype='datetime64[ns]', name='Date', length=60649, freq=None)

In [1631]:
df.values

array([['    NaN', '    NaN', '    NaN', '    NaN', '    NaN', '    NaN'],
       ['    NaN', '    NaN', '    NaN', '    NaN', '    NaN', '    NaN'],
       ['    NaN', '    NaN', '    NaN', '    NaN', '    NaN', '    NaN'],
       ...,
       ['  0.000', ' 28.190', '  57.87', '  2.043', ' 157.20', '   0.00'],
       ['  0.000', ' 27.670', '  60.88', '  2.485', ' 163.60', '   0.00'],
       ['  0.000', ' 26.850', '  66.20', '  2.691', ' 173.40', '   0.00']],
      dtype=object)

In [1632]:
df.columns

Index([' Ppt', ' Tair', ' RH', ' Wind speed', ' Wind direction', ' Srad   '], dtype='object')

In [1633]:
df.columns = df.columns.str.replace(' ','')
df.columns


Index(['Ppt', 'Tair', 'RH', 'Windspeed', 'Winddirection', 'Srad'], dtype='object')

In [1634]:
# Drop flag only for sm data
# df = df.drop(columns=['Flag'])


In [1635]:
# removing spaces from anywhere, and converting types to floats bcz currently in str?
df.values
df = df.map(lambda x: float(str(x).replace(' ', '')))
df


,Ppt,Tair,RH,Windspeed,Winddirection,Srad
Date,,,,,,
2014-10-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2021-08-31 20:00:00,0.0,29.94,52.58,1.237,149.7,0.06
2021-08-31 21:00:00,0.0,28.88,55.09,1.612,160.1,0.00
2021-08-31 22:00:00,0.0,28.19,57.87,2.043,157.2,0.00


In [1636]:
df.index # not in float val but makes sense

DatetimeIndex(['2014-10-01 00:00:00', '2014-10-01 01:00:00',
               '2014-10-01 02:00:00', '2014-10-01 03:00:00',
               '2014-10-01 04:00:00', '2014-10-01 05:00:00',
               '2014-10-01 06:00:00', '2014-10-01 07:00:00',
               '2014-10-01 08:00:00', '2014-10-01 09:00:00',
               ...
               '2021-08-31 15:00:00', '2021-08-31 16:00:00',
               '2021-08-31 17:00:00', '2021-08-31 18:00:00',
               '2021-08-31 19:00:00', '2021-08-31 20:00:00',
               '2021-08-31 21:00:00', '2021-08-31 22:00:00',
               '2021-08-31 23:00:00', '2021-09-01 00:00:00'],
              dtype='datetime64[ns]', name='Date', length=60649, freq=None)

In [1637]:
df.values # better, now all in float values

array([[    nan,     nan,     nan,     nan,     nan,     nan],
       [    nan,     nan,     nan,     nan,     nan,     nan],
       [    nan,     nan,     nan,     nan,     nan,     nan],
       ...,
       [  0.   ,  28.19 ,  57.87 ,   2.043, 157.2  ,   0.   ],
       [  0.   ,  27.67 ,  60.88 ,   2.485, 163.6  ,   0.   ],
       [  0.   ,  26.85 ,  66.2  ,   2.691, 173.4  ,   0.   ]])

In [1638]:
# print(df['SWC_5'].dtype) # all numeric types

In [1639]:
# automatically shows df plotted
%matplotlib inline

### UNCOMMENT IF YOU WANT TO SEE GRAPH OF EACH YEAR EACH VARIABLE

# # each year, each variable 
# years = [str(item) for item in range(2015, 2022)]

# # per years 
# for year in years:
#     # per variable/column
#     for col in df.columns:
#         # Resample data for each column by day and take the mean
#         df_yearly = df.loc[year, col].resample('D').mean().reset_index()

#         # line plot w *new* plotly
#         fig = px.line(df_yearly, x='Date', y=col, 
#                       title=f'{col} Daily Mean for {year}', 
#                       labels={'Date': 'Date', col: f'{col} Value'},
#                       width=1200, height=600)  # Equivalent to figsize=(12,6)
 #       fig.show()

In [1640]:
# # just 2018!

# # all years one graph
# years = [str(item) for item in range(2015, 2022)] 
# all_years_data = []

# # per year, only SWC_5 //so useless
# for year in years:
#     df_yearly = df.loc[year].SWC_5.resample('D').mean().reset_index() 
#     df_yearly['Year'] = year
#     all_years_data.append(df_yearly)
# combined_data_df = pd.concat(all_years_data)

# # line plotting
# fig = px.line(combined_data_df, x='Date', y='SWC_5', color='Year', 
#                  title='SWC_5 Daily Mean for All Years', 
#                  labels={'index': 'Date', 'SWC_5': 'SWC_5 Value'},
#                  width=1200, height=600)
# fig.show()

In [1641]:
years = [str(item) for item in range(2015, 2022)]  # 2015 to 2022

# Ppt
ppt_data = []
for year in years:
    if "Ppt" in df.columns:
        df_yearly = df.loc[year, "Ppt"].resample('D').mean().reset_index()
        df_yearly["Year"] = year  # Add year for coloring
        ppt_data.append(df_yearly)

# combine
if ppt_data:
    ppt_combined = pd.concat(ppt_data)
    fig = px.line(ppt_combined, x='Date', y='Ppt', color='Year', 
                  title="Daily Mean Precipitation (Ppt) from 2015 to 2022",
                  labels={'Date': 'Date', 'Ppt': 'Precipitation (mm)'},
                  width=1200, height=600)
    # fig.show()

In [1642]:
# SWC 
swc_cols = ["SWC_5", "SWC_10", "SWC_20", "SWC_50"]

# make one graph per year
for year in range(2015, 2022):  
    swc_data = []

    # show each swc
    for col in swc_cols:
        if col in df.columns:
            df_yearly = df.loc[str(year), col].resample('D').mean().reset_index()
            df_yearly["SWC Type"] = col 
            df_yearly.rename(columns={col: "SWC Value"}, inplace=True)
            swc_data.append(df_yearly)
    
    # putting all data togegher
    if swc_data:
        swc_combined = pd.concat(swc_data)
        fig = px.line(swc_combined, x='Date', y="SWC Value", color='SWC Type',
                      title=f"Daily Mean Soil Water Content (SWC) for {year}",
                      labels={'Date': 'Date', 'SWC Value': 'SWC (Soil Water Content)'},
                      width=1200, height=600)
        
        # fig.show()


In [1643]:
# T - copy and paste except for temperature
temp_cols = ["T_5", "T_10", "T_20", "T_50"]

# makes one graph per year
for year in range(2015, 2022): 
    temp_data = []
    
    for col in temp_cols:
        if col in df.columns:
            df_yearly = df.loc[str(year), col].resample('D').mean().reset_index()
            df_yearly["Temperature Type"] = col 
            df_yearly.rename(columns={col: "Temperature Value"}, inplace=True)
            temp_data.append(df_yearly)
    
    # combined all temp data
    if temp_data:
        temp_combined = pd.concat(temp_data)
        fig = px.line(temp_combined, x='Date', y="Temperature Value", color='Temperature Type',
                      title=f"Daily Mean Temperature (T) for {year}",
                      labels={'Date': 'Date', 'Temperature Value': 'Temperature (°C)'},
                      width=1200, height=600)
        
        # fig.show()


last graph, this is the subproblem part. gives the ppt beside temp and swc per year

In [1644]:
# columns
ppt_col = ["Ppt"] 
swc_cols = ["SWC_5", "SWC_10", "SWC_20", "SWC_50"]

# looping through years
for year in range(2015, 2022):  
    yearly_data = []

    # all columnbs
    for col in ppt_col + swc_cols:  
        if col in df.columns:
            df_yearly = df.loc[str(year), col].resample('D').mean().reset_index()
            df_yearly["Variable Type"] = col
            df_yearly.rename(columns={col: "Value"}, inplace=True)
            yearly_data.append(df_yearly)

    # combine all
    if yearly_data:
        combined_data = pd.concat(yearly_data)
        
        # line plot!!
        fig = px.line(combined_data, x='Date', y="Value", color='Variable Type',
                      title=f"Daily Mean of Ppt and SWC for {year}",
                      labels={'Date': 'Date', 'Value': 'Measurement'},
                      width=1200, height=600)
        
        # fig.show()


In [1645]:
# lets count the missing values
print("Missing Values Summary by Col")
for col in df.columns:
    print(col + ": " + str(df[col].isna().sum()))

Missing Values Summary by Col
Ppt: 1045
Tair: 1045
RH: 1045
Windspeed: 1045
Winddirection: 1045
Srad: 1177


### Missing Values Count & Percentages

In [1646]:
import pandas as pd

# Calculate total values, non-null values, and missing values per column
total_values = df.shape[0]  # Total rows in df
non_null_counts = df.count()
missing_counts = df.isna().sum()
missing_percentage = (missing_counts / total_values) * 100

# Calculate total missing values across the entire dataset
total_missing_values = missing_counts.sum()
total_possible_values = df.size  # Total elements in df (rows * columns)
total_missing_percentage = (total_missing_values / total_possible_values) * 100

# Create a summary DataFrame
summary_df = pd.DataFrame({
    "Total Values": total_values,
    "Non-null Values": non_null_counts,
    "Missing Values": missing_counts,
    "Missing (%)": missing_percentage.round(2)  # Round to 2 decimal places
})

# Append total missing percentage as a new row
summary_df.loc["TOTAL"] = [df.size, df.size - total_missing_values, total_missing_values, round(total_missing_percentage, 2)]

# Save to Excel
summary_df.to_excel("missing_data_summary.xlsx", sheet_name="Summary")

print("Missing data summary saved as 'missing_data_summary.xlsx'.")


Missing data summary saved as 'missing_data_summary.xlsx'.


In [1647]:
print(df.index.dtype)

datetime64[ns]


### Precisely counting missing values

In [1648]:
#lets count missing values, more precisely

for col in df.columns:
    # respample by year ('A' for annual, 'M' for monthly, 'W' for weekly, 'D' for daily)
    for freq in ['YE', 'ME', 'W', 'D']:  # Change around A, M, W, D as needed
        # Resample and count missing values using .isna().sum()
        missing = df[col].resample(freq).apply(lambda x: x.isna().sum())

        # Check if there are any missing values
        if missing.sum() > 0:
            non_zero_missing = missing[missing > 0]
            if not non_zero_missing.empty:
                print(f"{col} ({freq}): \n{non_zero_missing}\n")

print("End of all missing values")

Ppt (YE): 
Date
2014-12-31    1044
2019-12-31       1
Name: Ppt, dtype: int64

Ppt (ME): 
Date
2014-10-31    744
2014-11-30    300
2019-11-30      1
Name: Ppt, dtype: int64

Ppt (W): 
Date
2014-10-05    120
2014-10-12    168
2014-10-19    168
2014-10-26    168
2014-11-02    168
2014-11-09    168
2014-11-16     84
2019-11-24      1
Name: Ppt, dtype: int64

Ppt (D): 
Date
2014-10-01    24
2014-10-02    24
2014-10-03    24
2014-10-04    24
2014-10-05    24
2014-10-06    24
2014-10-07    24
2014-10-08    24
2014-10-09    24
2014-10-10    24
2014-10-11    24
2014-10-12    24
2014-10-13    24
2014-10-14    24
2014-10-15    24
2014-10-16    24
2014-10-17    24
2014-10-18    24
2014-10-19    24
2014-10-20    24
2014-10-21    24
2014-10-22    24
2014-10-23    24
2014-10-24    24
2014-10-25    24
2014-10-26    24
2014-10-27    24
2014-10-28    24
2014-10-29    24
2014-10-30    24
2014-10-31    24
2014-11-01    24
2014-11-02    24
2014-11-03    24
2014-11-04    24
2014-11-05    24
2014-11-06    2

In [1649]:
# Find missing values in SWC_20
missing_Tair = df[df["Tair"].isna()]

# Print count of missing values
print(f"Number of missing values in Tair: {missing_Tair.shape[0]}")

# Print timestamps of missing values
print("Timestamps of missing hours in Tair:")
print(missing_Tair.index)

Number of missing values in Tair: 1045
Timestamps of missing hours in Tair:
DatetimeIndex(['2014-10-01 00:00:00', '2014-10-01 01:00:00',
               '2014-10-01 02:00:00', '2014-10-01 03:00:00',
               '2014-10-01 04:00:00', '2014-10-01 05:00:00',
               '2014-10-01 06:00:00', '2014-10-01 07:00:00',
               '2014-10-01 08:00:00', '2014-10-01 09:00:00',
               ...
               '2014-11-13 03:00:00', '2014-11-13 04:00:00',
               '2014-11-13 05:00:00', '2014-11-13 06:00:00',
               '2014-11-13 07:00:00', '2014-11-13 08:00:00',
               '2014-11-13 09:00:00', '2014-11-13 10:00:00',
               '2014-11-13 11:00:00', '2019-11-19 09:00:00'],
              dtype='datetime64[ns]', name='Date', length=1045, freq=None)


In [1650]:
# # Count missing values per hour
# missing_per_hour = df.isna().resample('h').sum()

# # Count missing values per day
# missing_per_day = df.isna().resample('D').sum()

# # Print summary
# print("Missing Values Per Hour:")
# print(missing_per_hour)

# print("\nMissing Values Per Day:")
# print(missing_per_day)

# Check if SWC_20 exists in df
if "Tair" in df.columns:
    # Find days that have at least one missing value
    missing_days = df["Tair"].isna().resample('D').sum() > 0
    num_missing_days = missing_days.sum()  # Count days with missing values

    # Count total missing hours
    num_missing_hours = df["Tair"].isna().sum()

    # Print results
    print(f"Total days with missing Tair values: {num_missing_days}")
    print(f"Total missing Tair hours: {num_missing_hours}")
else:
    print("Tair column not found in the dataframe.")

Total days with missing Tair values: 45
Total missing Tair hours: 1045


## trial again for cooccurance -- newest version

In [1651]:
# Import & clean both datasets
sm_df = pd.read_csv('../../datasets/TX-Data/soil_station/SM_2.dat', sep=",", parse_dates=["Date"], index_col="Date") 
met_df = pd.read_csv('../../datasets/TX-Data/MET_station/MET_2.dat', sep=",", parse_dates=["Date"], index_col="Date") 

# Manually converting date column to datetime
sm_df.index = pd.to_datetime(sm_df.index, format='%m/%d/%y %H:%M')
met_df.index = pd.to_datetime(met_df.index, format='%m/%d/%y %H:%M')

# Drop NaN rows (still doing it anyway)
sm_df.dropna()
met_df.dropna()

# Strip whitespace from column names
sm_df.columns = sm_df.columns.str.replace(' ','')
met_df.columns = met_df.columns.str.replace(' ','')

# Rename "Ppt" columns before merging
sm_df = sm_df.rename(columns={"Ppt": "Ppt_sm"})
met_df = met_df.rename(columns={"Ppt": "Ppt_met"})

# Convert values to float using original method
sm_df = sm_df.map(lambda x: float(str(x).replace(' ', '')))
met_df = met_df.map(lambda x: float(str(x).replace(' ', '')))

# Drop 'Flag' column if present
if 'Flag' in sm_df.columns:
    sm_df = sm_df.drop(columns=['Flag'])

/var/folders/kf/3zgjks2j09s7fg110wp13m6h0000gn/T/ipykernel_20376/4231864724.py:2: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/var/folders/kf/3zgjks2j09s7fg110wp13m6h0000gn/T/ipykernel_20376/4231864724.py:3: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [1652]:

# Check if Tair exists in df
if "Tair" in df.columns:
    # Find days that have at least one missing value
    missing_days = df["Tair"].isna().resample('D').sum() > 0
    num_missing_days = missing_days.sum()  # Count days with missing values

    # Count total missing hours
    num_missing_hours = df["Tair"].isna().sum()

    # Print results
    print(f"Total days with missing Tair values: {num_missing_days}")
    print(f"Total missing Tair hours: {num_missing_hours}")
else:
    print("Tair column not found in the dataframe.")

Total days with missing Tair values: 45
Total missing Tair hours: 1045


option 2 -- ACCURATE VERSION

In [1653]:
# Initialize an empty matrix to hold missing value counts
missing_matrix = pd.DataFrame(index=sm_df.columns.tolist() + met_df.columns.tolist(),
                              columns=sm_df.columns.tolist() + met_df.columns.tolist(),
                              dtype=int).fillna(0)

# Iterate through each pair of columns from both sm_df and met_df
for col1 in sm_df.columns.tolist() + met_df.columns.tolist():
    for col2 in sm_df.columns.tolist() + met_df.columns.tolist():
        # Count the missing values where both columns have missing values
        if col1 in sm_df.columns and col2 in sm_df.columns:
            missing_count = (sm_df[col1].isna() & sm_df[col2].isna()).sum()
        elif col1 in met_df.columns and col2 in met_df.columns:
            missing_count = (met_df[col1].isna() & met_df[col2].isna()).sum()
        elif col1 in sm_df.columns and col2 in met_df.columns:
            missing_count = (sm_df[col1].isna() & met_df[col2].isna()).sum()
        elif col1 in met_df.columns and col2 in sm_df.columns:
            missing_count = (met_df[col1].isna() & sm_df[col2].isna()).sum()

        # Store the result in the matrix
        missing_matrix.loc[col1, col2] = missing_count

# Print the missing value comparison matrix
print("\nMissing Value Comparison between sm_data and met_data (separate columns):")
print(missing_matrix)

# Export the matrix to an Excel file
missing_matrix.to_excel('co_occurrence_numerical.xlsx')



Missing Value Comparison between sm_data and met_data (separate columns):
               Ppt_sm   SWC_5  SWC_10  SWC_20  SWC_50     T_5    T_10    T_20  \
Ppt_sm         2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0   
SWC_5          2795.0  2797.0  2796.0  2796.0  2795.0  2795.0  2795.0  2795.0   
SWC_10         2795.0  2796.0  2796.0  2796.0  2795.0  2795.0  2795.0  2795.0   
SWC_20         2795.0  2796.0  2796.0  2796.0  2795.0  2795.0  2795.0  2795.0   
SWC_50         2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0   
T_5            2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0   
T_10           2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0   
T_20           2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0   
T_50           2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0  2795.0   
Ppt_met        2075.0  2075.0  2075.0  2075.0  2075.0  2075.0  2075.0  2075.0   
Tair           2075.0  2075.0  207

In [1654]:
# Initialize an empty matrix to hold missing value counts and percentages
missing_matrix = pd.DataFrame(index=sm_df.columns.tolist() + met_df.columns.tolist(),
                              columns=sm_df.columns.tolist() + met_df.columns.tolist(),
                              dtype=float).fillna(0)

# Get total missing values for each column
sm_missing_counts = sm_df.isna().sum()
met_missing_counts = met_df.isna().sum()

# Iterate through each pair of columns from both sm_df and met_df
for col1 in sm_df.columns.tolist() + met_df.columns.tolist():
    for col2 in sm_df.columns.tolist() + met_df.columns.tolist():
        # Count the missing values where both columns have missing values
        if col1 in sm_df.columns and col2 in sm_df.columns:
            missing_count = (sm_df[col1].isna() & sm_df[col2].isna()).sum()
        elif col1 in met_df.columns and col2 in met_df.columns:
            missing_count = (met_df[col1].isna() & met_df[col2].isna()).sum()
        elif col1 in sm_df.columns and col2 in met_df.columns:
            missing_count = (sm_df[col1].isna() & met_df[col2].isna()).sum()
        elif col1 in met_df.columns and col2 in sm_df.columns:
            missing_count = (met_df[col1].isna() & sm_df[col2].isna()).sum()

        # Calculate total missing values for the two columns
        total_missing = sm_missing_counts[col1] + met_missing_counts[col2] if col1 in sm_df.columns and col2 in met_df.columns else sm_missing_counts[col1] + met_missing_counts[col2]

        # Calculate the percentage of missing data overlap
        if total_missing > 0:
            missing_percentage = (missing_count / total_missing) * 100
        else:
            missing_percentage = 0

        # Store the result in the matrix
        missing_matrix.loc[col1, col2] = missing_percentage

# Format the matrix values as scientific notation
missing_matrix = missing_matrix.applymap(lambda x: f'{x:.2e}' if isinstance(x, (int, float)) else x)

# Print the missing value comparison matrix
print("\nMissing Value Comparison between sm_data and met_data (percentage of total missing values):")
print(missing_matrix)

# Export the matrix to an Excel file
missing_matrix.to_excel('co_occurrence_percentages_scientific_notation.xlsx')


KeyError: 'Ppt_sm'

### Missing Values information
- Have some check – how many fields are nan? found!!
- What were missing values? 
- How to impute missing data? 
- What model do you use to impute data? LSTM/SARIMA!!
- Find soil moisture only or soil temp only and figure out how to replace it?


#### Brainstorming ways to fill in missing values (April,May, June 2018)

- Forward/back fill: for continuity                                   //no ! bad idea
- Mean / median / mode: for relatively stable data without strong seasonality.      // no! bad idea
- Seasonal Interpolation                                                        // maybe... gap is too big? maybe after prediction
                                                                                // can interpolate between smaller gaps, but not
                          

                                                      // good for big picture. Use prediction first
                                                                                // and can do polynomial/spline (maybe not linear)
- Predictive Modeling: For datasets with additional context features.           // yes, v good idea

Overall ideas: 
1. Predictive using past seasonal trends, sarima sounds good. then interpolation on top with v small windows of time

## BELOW Finding Invalid Values

In [ ]:
df

,Ppt,Tair,RH,Windspeed,Winddirection,Srad
Date,,,,,,
2014-10-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2014-10-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2021-08-31 20:00:00,0.0,29.94,52.58,1.237,149.7,0.06
2021-08-31 21:00:00,0.0,28.88,55.09,1.612,160.1,0.00
2021-08-31 22:00:00,0.0,28.19,57.87,2.043,157.2,0.00


In [ ]:
import pandas as pd
import numpy as np

# Define valid thresholds
swc_cols = ["SWC_5", "SWC_10", "SWC_20", "SWC_50"]
wind_speed_col = "Windspeed"
wind_dir_col = "Winddirection"
tair_col = "Tair"
rh_col = "RH"
srad_col = "Srad"

# Function to find invalid and valid data
def analyze_data_validity(df):
    invalid_data = {}
    valid_data = {}

    # Soil moisture content (SWC) between 0.0 and 1.0
    for col in swc_cols:
        if col in df:
            invalid_data[col] = df[~df[col].between(0.0, 1.0)][[col]]
            valid_data[col] = df[df[col].between(0.0, 1.0)][[col]]

    # Air temperature (Tair) constraints
    if tair_col in df:
        tair_diff_1h = df[tair_col].diff().abs()  # Difference from previous hour
        tair_diff_12h = df[tair_col].diff(periods=12).abs()  # Over 12 hours

        invalid_data[tair_col] = df[
            (~df[tair_col].between(-50, 50)) |  # Example local extreme range
            (tair_diff_1h > 5) |  # Temp shouldn't jump more than 5°C in 1h
            (tair_diff_12h >= 0.5)  # Should vary at least 0.5°C over 12h
        ][[tair_col]]

        valid_data[tair_col] = df[
            (df[tair_col].between(-50, 50)) & 
            (tair_diff_1h <= 5) & 
            (tair_diff_12h < 0.5)
        ][[tair_col]]

    # Relative Humidity (RH) based on dew point temp
    if rh_col in df and tair_col in df:
        dew_point_temp = df[tair_col] - ((100 - df[rh_col]) / 5)  # Approximate dew point
        rh_diff_1h = dew_point_temp.diff().abs()

        invalid_data[rh_col] = df[
            (dew_point_temp > df[tair_col]) |  # Dew point shouldn't exceed air temp
            (rh_diff_1h > 5) |  # Dew point shouldn't change >5°C per hour
            ((dew_point_temp - df[tair_col]).rolling(12).max() < 0)  # Check 12-hour trend
        ][[rh_col]]

        valid_data[rh_col] = df[
            (dew_point_temp <= df[tair_col]) & 
            (rh_diff_1h <= 5) & 
            ((dew_point_temp - df[tair_col]).rolling(12).max() >= 0)
        ][[rh_col]]

    # Wind speed (0 m/s ≥ WS ≤ 25 m/s, and variations over time)
    if wind_speed_col in df:
        ws_diff_3h = df[wind_speed_col].diff(periods=3).abs()
        ws_diff_12h = df[wind_speed_col].diff(periods=12).abs()

        invalid_data[wind_speed_col] = df[
            (~df[wind_speed_col].between(0, 25)) |
            (ws_diff_3h >= 0.1) |
            (ws_diff_12h >= 0.5)
        ][[wind_speed_col]]

        valid_data[wind_speed_col] = df[
            (df[wind_speed_col].between(0, 25)) & 
            (ws_diff_3h < 0.1) & 
            (ws_diff_12h < 0.5)
        ][[wind_speed_col]]

    # Wind direction (0°≥ WD ≤ 360°, and variations over time)
    if wind_dir_col in df:
        wd_diff_3h = df[wind_dir_col].diff(periods=3).abs()

        invalid_data[wind_dir_col] = df[
            (~df[wind_dir_col].between(0, 360)) |
            (wd_diff_3h >= 1)
        ][[wind_dir_col]]

        valid_data[wind_dir_col] = df[
            (df[wind_dir_col].between(0, 360)) & 
            (wd_diff_3h < 1)
        ][[wind_dir_col]]

    # Solar Radiation (Nighttime SR = 0, Daytime SR < max)
    if srad_col in df:
        invalid_data[srad_col] = df[
            ((df.index.hour < 6) | (df.index.hour > 18)) & (df[srad_col] > 0)  # Nighttime should be 0
        ][[srad_col]]

        valid_data[srad_col] = df[
            (((df.index.hour >= 6) & (df.index.hour <= 18)) | (df[srad_col] == 0))
        ][[srad_col]]

    # Convert results into DataFrames
    invalid_df = pd.concat(invalid_data, axis=1)
    valid_df = pd.concat(valid_data, axis=1)

    return invalid_df, valid_df

# Run function to find invalid and valid data
invalid_data_df, valid_data_df = analyze_data_validity(df)

# Print full invalid data
print("\nInvalid Data Points:")
print(invalid_data_df)

# Print summary of how many invalid points exist for each column
print("\nSummary of Invalid Data:")
print(invalid_data_df.count())

# Print summary of how many valid points exist for each column
print("\nSummary of Valid Data:")
print(valid_data_df.count())



Invalid Data Points:
                      Tair     RH Windspeed Winddirection  Srad
                      Tair     RH Windspeed Winddirection  Srad
Date                                                           
2014-10-01 00:00:00    NaN    NaN       NaN           NaN   NaN
2014-10-01 01:00:00    NaN    NaN       NaN           NaN   NaN
2014-10-01 02:00:00    NaN    NaN       NaN           NaN   NaN
2014-10-01 03:00:00    NaN    NaN       NaN           NaN   NaN
2014-10-01 04:00:00    NaN    NaN       NaN           NaN   NaN
...                    ...    ...       ...           ...   ...
2021-08-31 20:00:00  29.94  52.58     1.237         149.7  0.06
2021-08-31 21:00:00  28.88  55.09     1.612         160.1   NaN
2021-08-31 22:00:00  28.19  57.87     2.043         157.2   NaN
2021-08-31 23:00:00  27.67  60.88     2.485         163.6   NaN
2021-09-01 00:00:00  26.85  66.20     2.691         173.4   NaN

[60617 rows x 5 columns]

Summary of Invalid Data:
Tair           Tair           